# ML-02: Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Succypet0/flyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order**, each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane, or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane**: CTR / Engagement Opportunity Scoring.

**Why this lane**: Content writers and SEO publishers invest significant time creating articles that rank on Google Page 1 but frequently suffer from low click-through rates (CTR) and poor on-page reader engagement (scroll depth and engaged sessions). Optimizing existing high-visibility articles through targeted title/meta snippet rewrites and on-page structural improvements (scannable headings, intro hooks, clear formatting) delivers far higher ROI than writing new articles from scratch. This project builds an opportunity scoring model to pinpoint published articles where the gap between search visibility and user interaction is widest.

In [ ]:
# Lane definition confirmation
LANE = "CTR / Engagement Opportunity Scoring"
TARGET_PERSONA = "Content Writer / SEO Publisher"

print(f"Active Project Lane: {LANE}")
print(f"Target Persona: {TARGET_PERSONA}")


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**1. Decision**: Which published articles should a content writer or SEO editor prioritize for structural revisions (title/snippet rewrites, intro hooks, layout scannability) in their upcoming editorial sprint?

**2. Actor and Action**: Content writers and SEO publishers. They will take the top-ranked articles from the opportunity queue and perform targeted optimizations: refining SERP title tags and meta descriptions to improve CTR, and restructuring subheadings, media, and opening paragraphs to improve scroll depth and reader retention.

**3. Cost of a Wrong Call**: A false positive recommendation wastes limited editorial hours restructuring an article that either already has optimal CTR for its query intent or lacks real search demand. A false negative leaves high-visibility decaying pages unoptimized, leaking organic traffic and audience engagement.

**4. Why Data / ML Helps**: Expected CTR and engagement vary non-linearly with search position, query intent, content format, and word count. A static heuristic (such as a flat CTR < 1% rule) fails because position 1 has completely different expected benchmarks than position 8. A learned scoring model accounts for these multivariate factors to surface true underperformers.

In [ ]:
# Core framing parameters
decision = "Prioritize existing articles for title/snippet rewrites and on-page structural restructuring"
actor = "Content Writer / SEO Publisher"
action = "Rewrite SERP snippets and restructure page headings/layout"
cost_of_error = "Wasted editorial hours on low-impact pages vs lost traffic on decaying high-visibility assets"

print(f"Decision: {decision}")
print(f"Actor: {actor}")
print(f"Action: {action}")
print(f"Cost of Error: {cost_of_error}")


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

An exploratory audit of the starter dataset (`data/raw/content_refresh_anonymized.csv`) reveals four key empirical numbers supporting the value of this lane:

1. **Valid Search Base**: 28,795 out of 30,000 articles (96.0%) have active ranking position data (`avg_position > 0`).
2. **Page 1 Visibility Pool**: 12,891 articles rank on Google Page 1 (positions 1.0 to 10.0), capturing **96,819,454 search impressions**.
3. **The Underperformance Gap**: **7,174 Page 1 articles** with high visibility (>= 500 impressions) underperform with a CTR below 1.0%, accounting for **91,866,225 impressions** where snippet and structural improvements can recover lost clicks.
4. **Engagement Baseline**: Median scroll rate is 5.0% across the corpus, indicating substantial room for on-page structure and scannability improvements.

In [ ]:
import os
import pandas as pd
import numpy as np

# Robust data path handling for Colab and local execution
data_candidates = [
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv"
]
data_path = next((p for p in data_candidates if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Starter dataset not found in expected paths.")

df = pd.read_csv(data_path)

# Metric 1: Valid search rankings
valid_rankings = df[df["avg_position"] > 0]
total_rows = len(df)
valid_count = len(valid_rankings)

# Metric 2: Page 1 rankings (avg_position between 1.0 and 10.0)
page_1 = valid_rankings[(valid_rankings["avg_position"] >= 1.0) & (valid_rankings["avg_position"] <= 10.0)]
p1_count = len(page_1)
p1_impressions = int(page_1["impressions_90d"].sum())

# Metric 3: Underperforming high-impression Page 1 articles (impressions >= 500 and CTR < 1.0%)
high_opp = page_1[(page_1["impressions_90d"] >= 500) & (page_1["ctr"] < 1.0)]
high_opp_count = len(high_opp)
high_opp_impressions = int(high_opp["impressions_90d"].sum())

# Metric 4: Median engagement and scroll rate
med_scroll = float(df["scroll_rate"].median())

print(f"Total articles in slice: {total_rows:,}")
print(f"Articles with valid ranking position: {valid_count:,} ({valid_count/total_rows*100:.1f}%)")
print(f"Page 1 articles (pos 1-10): {p1_count:,} capturing {p1_impressions:,} impressions")
print(f"High-opportunity Page 1 articles (>=500 impr, CTR < 1%): {high_opp_count:,} capturing {high_opp_impressions:,} impressions")
print(f"Corpus median scroll rate: {med_scroll:.1f}%")


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In alignment with FlyRank claim discipline and honest modeling practices:

**What this work CAN claim**:
- **Observed correlations**: Historical relationships between ranking position, content type, word count, CTR, and reader engagement metrics.
- **Directional prioritization**: Relative ranking of which articles exhibit the largest statistical gap between actual vs expected performance.
- **Decision support**: Actionable recommendations that guide editorial sprint planning and content restructuring workflows.

**What this work CANNOT claim**:
- **Causal guarantees**: We cannot claim that rewriting a title tag will guarantee a specific CTR jump or higher ranking.
- **Predicting Google's algorithm**: We do not claim to reverse engineer or predict Google Search ranking algorithms.
- **Deterministic traffic outcomes**: Search traffic is subject to market competition, seasonality, and shifting searcher intent.

In [ ]:
# Claim boundaries definition
allowed_claims = [
    "Observed correlations across search position and engagement",
    "Directional priority scoring for underperforming content",
    "Decision-support recommendations for editorial planning"
]
forbidden_claims = [
    "Causal guarantees of ranking or traffic increases",
    "Predicting Google's ranking algorithm",
    "Deterministic traffic or conversion promises"
]

print("Allowed Claims:")
for claim in allowed_claims:
    print(f"  [+] {claim}")

print("\nForbidden Claims:")
for claim in forbidden_claims:
    print(f"  [-] {claim}")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.